# 🌍 Topic 19 — Global Development Data Investigation
### IT for Youth Ghana · Data Analytics with Python

---

**Dataset:** Gapminder five-year country indicators  
**Local file:** `investigate_dataset/data/gapminder_five_year.csv`  
**Source:** https://raw.githubusercontent.com/plotly/datasets/master/gapminderDataFiveYear.csv

**How to use this notebook**
- Run cells from top to bottom with **Shift + Enter**
- Complete the **🔨 Try It** cells before moving on
- This lesson uses `pandas`, `numpy`, and `matplotlib`

**Assignment**
- Complete [Topic 19 Assignment — Gapminder Development Analysis](assignments/topic19_assignment_gapminder_analysis.ipynb).

In the original investigation notebook, the analysis focused on a movie dataset. In this lesson, we will investigate a real public CSV dataset about countries, continents, population, life expectancy, and GDP per capita.
**Before analysis**
- Read [Topic 19A — Data Gathering and Understanding a Dataset](topic19a_data_gathering_and_understanding.ipynb) before starting this investigation.
- Use [Appendix A — Common Data Sources](topic19b_appendix_common_data_sources.ipynb) when choosing datasets for projects.
- Use [Appendix B — Reading Dataset Documentation](topic19c_appendix_reading_dataset_documentation.ipynb) when evaluating a new dataset.


---
## 🗺️ The Full Analysis Process

A complete data investigation follows the same path you have practised across earlier topics:

| Stage | What we do here |
|-------|-----------------|
| 1. Data Gathering | Load the downloaded public CSV |
| 2. Data Wrangling | Check quality and create clearer columns |
| 3. EDA | Explore distributions, groups, and trends |
| 4. Analysis | Answer focused questions about development patterns |
| 5. Visualisation | Use charts to make patterns visible |
| 6. Insights | Communicate findings and limitations |

The goal is not just to produce code. The goal is to tell a careful, evidence-based story from data.


---
## 📂 Stage 1 — Data Gathering

The dataset was downloaded from the internet and saved locally. We load it using `pd.read_csv()`.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

DATA_PATH = "investigate_dataset/data/gapminder_five_year.csv"
SOURCE_URL = "https://raw.githubusercontent.com/plotly/datasets/master/gapminderDataFiveYear.csv"

df = pd.read_csv(DATA_PATH)

print("Dataset loaded successfully")
print("Source:", SOURCE_URL)
print("Shape:", df.shape)
print()
print(df.head())


In [ ]:
print("Columns:")
print(list(df.columns))
print()
print("Data types:")
print(df.dtypes)


### Column Meaning

| Column | Meaning |
|--------|---------|
| `country` | Country name |
| `year` | Year of observation |
| `pop` | Population |
| `continent` | Continent |
| `lifeExp` | Life expectancy in years |
| `gdpPercap` | GDP per person |


---
## 🧹 Stage 2 — Data Wrangling

We check whether the dataset is clean enough for analysis, then create clearer column names and calculated fields.


In [ ]:
print("Missing values:")
print(df.isna().sum())
print()
print("Duplicate rows:", df.duplicated().sum())
print("Countries:", df["country"].nunique())
print("Continents:", df["continent"].nunique())
print("Year range:", df["year"].min(), "to", df["year"].max())


In [ ]:
df = df.rename(columns={
    "pop": "population",
    "lifeExp": "life_expectancy",
    "gdpPercap": "gdp_per_capita",
})

df["population_millions"] = df["population"] / 1_000_000
df["total_gdp"] = df["population"] * df["gdp_per_capita"]
df["total_gdp_billions"] = df["total_gdp"] / 1_000_000_000

print(df.head())


In [ ]:
print("Value range checks")
print("-" * 40)
print("Life expectancy:", df["life_expectancy"].min(), "to", df["life_expectancy"].max())
print("GDP per capita :", round(df["gdp_per_capita"].min(), 2), "to", round(df["gdp_per_capita"].max(), 2))
print("Population     :", df["population"].min(), "to", df["population"].max())


**🔨 Try It 1 — Data Audit**

Write code to answer:

1. How many rows are there for each continent?
2. How many records does each country have?
3. Are there countries with fewer than 12 records?


In [ ]:
# 🔨 TRY IT 1

# 1. Rows per continent
# YOUR CODE HERE

# 2. Records per country
# YOUR CODE HERE

# 3. Countries with fewer than 12 records
# YOUR CODE HERE


---
## 🔍 Stage 3 — Exploratory Data Analysis

EDA helps us understand what is in the dataset before making claims.


In [ ]:
print("Numeric summary:")
print(df[["year", "population_millions", "life_expectancy", "gdp_per_capita", "total_gdp_billions"]].describe().round(2))


In [ ]:
latest_year = df["year"].max()
latest = df[df["year"] == latest_year].copy()

print("Latest year:", latest_year)
print("Countries in latest year:", len(latest))
print()
print(latest["continent"].value_counts())


In [ ]:
continent_summary = latest.groupby("continent").agg(
    countries=("country", "count"),
    total_population_millions=("population_millions", "sum"),
    avg_life_expectancy=("life_expectancy", "mean"),
    median_gdp_per_capita=("gdp_per_capita", "median"),
    total_gdp_billions=("total_gdp_billions", "sum"),
).sort_values("avg_life_expectancy", ascending=False)

print(continent_summary.round(2))


In [ ]:
ghana = df[df["country"] == "Ghana"].copy()
africa = df[df["continent"] == "Africa"].copy()

africa_by_year = africa.groupby("year").agg(
    africa_avg_life_expectancy=("life_expectancy", "mean"),
    africa_median_gdp_per_capita=("gdp_per_capita", "median"),
)

ghana_compare = ghana.merge(africa_by_year, on="year")
ghana_compare["life_expectancy_gap"] = ghana_compare["life_expectancy"] - ghana_compare["africa_avg_life_expectancy"]

print(ghana_compare[["year", "life_expectancy", "africa_avg_life_expectancy", "life_expectancy_gap", "gdp_per_capita"]].round(2))


**🔨 Try It 2 — EDA Practice**

Write code to answer:

1. Which country had the highest life expectancy in 2007?
2. Which African country had the highest GDP per capita in 2007?
3. Which continent had the largest total population in 2007?


In [ ]:
# 🔨 TRY IT 2

# YOUR CODE HERE


---
## 📊 Stage 4 — Visualisation

Tables are useful, but charts make trends and comparisons easier to see.


In [ ]:
continent_trend = df.groupby(["year", "continent"])["life_expectancy"].mean().reset_index()

plt.figure(figsize=(10, 6))
for continent in continent_trend["continent"].unique():
    subset = continent_trend[continent_trend["continent"] == continent]
    plt.plot(subset["year"], subset["life_expectancy"], marker="o", label=continent)

plt.title("Average Life Expectancy by Continent, 1952-2007")
plt.xlabel("Year")
plt.ylabel("Average life expectancy")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()


In [ ]:
life_2007 = latest.groupby("continent")["life_expectancy"].mean().sort_values()

plt.figure(figsize=(9, 5))
life_2007.plot(kind="barh", color="seagreen")
plt.title("Average Life Expectancy by Continent in 2007")
plt.xlabel("Average life expectancy")
plt.ylabel("Continent")
plt.grid(axis="x", alpha=0.3)
plt.show()


In [ ]:
plt.figure(figsize=(10, 6))
for continent in latest["continent"].unique():
    subset = latest[latest["continent"] == continent]
    plt.scatter(
        subset["gdp_per_capita"],
        subset["life_expectancy"],
        s=subset["population_millions"] * 2,
        alpha=0.6,
        label=continent,
    )

plt.xscale("log")
plt.title("GDP per Capita vs Life Expectancy, 2007")
plt.xlabel("GDP per capita (log scale)")
plt.ylabel("Life expectancy")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()


In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(ghana_compare["year"], ghana_compare["life_expectancy"], marker="o", label="Ghana")
plt.plot(ghana_compare["year"], ghana_compare["africa_avg_life_expectancy"], marker="o", label="Africa average")

plt.title("Ghana Life Expectancy Compared with Africa Average")
plt.xlabel("Year")
plt.ylabel("Life expectancy")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()


**🔨 Try It 3 — Visualisation Practice**

Create one chart of your own:

- Ghana GDP per capita over time
- Total population by continent in 2007
- Top 10 African countries by life expectancy in 2007


In [ ]:
# 🔨 TRY IT 3

# YOUR CODE HERE


---
## 📈 Stage 5 — Focused Analysis Questions

We now answer specific questions using grouped summaries, sorting, filtering, and correlation.


In [ ]:
life_by_continent_year = df.groupby(["continent", "year"])["life_expectancy"].mean().reset_index()

start = life_by_continent_year[life_by_continent_year["year"] == 1952][["continent", "life_expectancy"]]
end = life_by_continent_year[life_by_continent_year["year"] == 2007][["continent", "life_expectancy"]]

improvement = start.merge(end, on="continent", suffixes=("_1952", "_2007"))
improvement["change"] = improvement["life_expectancy_2007"] - improvement["life_expectancy_1952"]
improvement = improvement.sort_values("change", ascending=False)

print("Life expectancy improvement by continent:")
print(improvement.round(2))


In [ ]:
top_life_2007 = latest.sort_values("life_expectancy", ascending=False).head(10)

print("Top 10 countries by life expectancy in 2007:")
print(top_life_2007[["country", "continent", "life_expectancy", "gdp_per_capita"]].round(2).to_string(index=False))


In [ ]:
corr_2007 = latest["gdp_per_capita"].corr(latest["life_expectancy"])

print(f"Correlation between GDP per capita and life expectancy in 2007: {corr_2007:.3f}")
print("A positive value means higher GDP per capita is generally associated with higher life expectancy.")


In [ ]:
ghana_1952 = ghana[ghana["year"] == 1952].iloc[0]
ghana_2007 = ghana[ghana["year"] == 2007].iloc[0]

ghana_life_change = ghana_2007["life_expectancy"] - ghana_1952["life_expectancy"]
ghana_gdp_change_pct = (ghana_2007["gdp_per_capita"] / ghana_1952["gdp_per_capita"] - 1) * 100
ghana_pop_change_pct = (ghana_2007["population"] / ghana_1952["population"] - 1) * 100

print("Ghana change from 1952 to 2007")
print("-" * 40)
print(f"Life expectancy change : {ghana_life_change:.1f} years")
print(f"GDP per capita change  : {ghana_gdp_change_pct:.1f}%")
print(f"Population change      : {ghana_pop_change_pct:.1f}%")


**🔨 Try It 4 — Analysis Practice**

Answer two of these with code:

1. Which African country improved the most in life expectancy from 1952 to 2007?
2. Which continent had the widest GDP per capita spread in 2007?
3. Which country had the largest population in each continent in 2007?


In [ ]:
# 🔨 TRY IT 4

# YOUR CODE HERE


---
## 💡 Stage 6 — Insights and Communication

Data analysis becomes useful when we translate numbers into clear messages.


In [ ]:
top_improving_continent = improvement.iloc[0]
highest_life_country = top_life_2007.iloc[0]

insights = [
    f"{top_improving_continent['continent']} had the largest average life expectancy improvement, rising by "
    f"{top_improving_continent['change']:.1f} years from 1952 to 2007.",

    f"In 2007, {highest_life_country['country']} had the highest life expectancy in the dataset "
    f"at {highest_life_country['life_expectancy']:.1f} years.",

    f"GDP per capita and life expectancy had a positive relationship in 2007 "
    f"(correlation = {corr_2007:.2f}), though income is not the only factor.",

    f"Ghana's life expectancy increased by {ghana_life_change:.1f} years between 1952 and 2007, "
    f"while its population grew by {ghana_pop_change_pct:.0f}%.",
]

print("KEY INSIGHTS")
print("=" * 60)
for i, insight in enumerate(insights, 1):
    print(f"{i}. {insight}")


---
## 📤 Export a Summary Table


In [ ]:
export_summary = df.groupby(["continent", "year"]).agg(
    countries=("country", "count"),
    total_population_millions=("population_millions", "sum"),
    avg_life_expectancy=("life_expectancy", "mean"),
    median_gdp_per_capita=("gdp_per_capita", "median"),
    total_gdp_billions=("total_gdp_billions", "sum"),
).reset_index()

OUTPUT_PATH = "investigate_dataset/data/gapminder_continent_summary.csv"
export_summary.to_csv(OUTPUT_PATH, index=False)

print("Summary exported to:", OUTPUT_PATH)
print(export_summary.head())


---
## 🎯 Reflection

1. Which chart helped you understand the dataset best? Why?
2. What is one insight about Ghana that stood out to you?
3. What is one limitation of this dataset?
4. What extra column would make the analysis stronger?


**Your answers:**

1. *(type your answer here)*

2. *(type your answer here)*

3. *(type your answer here)*

4. *(type your answer here)*


---
## ✅ What You Covered

| Skill | Tool |
|-------|------|
| Load a public CSV | `pd.read_csv()` |
| Audit data quality | `.isna()`, `.duplicated()`, `.dtypes` |
| Create calculated columns | arithmetic with pandas columns |
| Explore patterns | `.describe()`, `.groupby()`, `.sort_values()` |
| Visualise data | `matplotlib` |
| Analyse relationships | `.corr()` and scatter plots |
| Export results | `.to_csv(index=False)` |

**Useful resources**
- Dataset source: https://raw.githubusercontent.com/plotly/datasets/master/gapminderDataFiveYear.csv
- pandas visualisation docs: https://pandas.pydata.org/docs/user_guide/visualization.html
- matplotlib pyplot tutorial: https://matplotlib.org/stable/tutorials/pyplot.html
